<a href="https://colab.research.google.com/github/SyedTahfim/Credit-Scorecard/blob/main/credit_scorecard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### __Import Libraries and Data__

In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import kagglehub
path = kagglehub.dataset_download("rikdifos/credit-card-approval-prediction")

Using Colab cache for faster access to the 'credit-card-approval-prediction' dataset.


**There are two data sets, one holds info about the applicants, and the other holds info about repayment.**

In [2]:
app_data = pd.read_csv(os.path.join(path, 'application_record.csv'))
credit_data = pd.read_csv(os.path.join(path, 'credit_record.csv'))

##### __Let's check the structure of the data__

In [ ]:
app_data.info()

In [ ]:
credit_data.info()

In [ ]:
credit_data.head()

### __Data Pre-processing__

**Before we merge the two data sets, we need to transform the credit_record data set to a format where there will be one record per applicant and the target will variable indicates whether the applicant defaulted.**

In [9]:
months_overdue = ["2","3","4","5"]
credit_data['IsDefault'] = credit_data['STATUS'].apply(lambda x: 1 if x in months_overdue else 0)
credit_data_transformed = credit_data.groupby('ID')['IsDefault'].max().reset_index()

In [11]:
credit_data_transformed['IsDefault'].value_counts()

,count
IsDefault,
0,45318
1,667


##### __Merge the two datasets__

In [18]:
df = app_data.merge(credit_data_transformed, on='ID', how='inner')
df = df.drop('ID',axis=1)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CODE_GENDER          36457 non-null  object 
 1   FLAG_OWN_CAR         36457 non-null  object 
 2   FLAG_OWN_REALTY      36457 non-null  object 
 3   CNT_CHILDREN         36457 non-null  int64  
 4   AMT_INCOME_TOTAL     36457 non-null  float64
 5   NAME_INCOME_TYPE     36457 non-null  object 
 6   NAME_EDUCATION_TYPE  36457 non-null  object 
 7   NAME_FAMILY_STATUS   36457 non-null  object 
 8   NAME_HOUSING_TYPE    36457 non-null  object 
 9   DAYS_BIRTH           36457 non-null  int64  
 10  DAYS_EMPLOYED        36457 non-null  int64  
 11  FLAG_MOBIL           36457 non-null  int64  
 12  FLAG_WORK_PHONE      36457 non-null  int64  
 13  FLAG_PHONE           36457 non-null  int64  
 14  FLAG_EMAIL           36457 non-null  int64  
 15  OCCUPATION_TYPE      25134 non-null 

##### __Null Values__

In [21]:
df.isna().sum()

,0
index,0
CODE_GENDER,0
FLAG_OWN_CAR,0
FLAG_OWN_REALTY,0
CNT_CHILDREN,0
AMT_INCOME_TOTAL,0
NAME_INCOME_TYPE,0
NAME_EDUCATION_TYPE,0
NAME_FAMILY_STATUS,0
NAME_HOUSING_TYPE,0


*Dropping OCCUPATION_TYPE variable because 31% of the rows have missing values.*

In [20]:
df = df.drop('OCCUPATION_TYPE', axis=1).reset_index()

**Let's separate numerical and categorical columns**

In [24]:
num_cols = df.select_dtypes(include=[int,float]).columns.tolist()  # Numerical Columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()  # Categorical Columns